# 第 5 章 · Tool Use（Google Search / ADK）

`google_search` 是**内建工具**：搜索在 Google 侧执行，`tools=[]` 里放的是框架导出的工具对象，
不是你写的函数。

所以「模型到底调没调工具」代码里看不出来——只能查 `grounding_metadata`。
这也正是本章那条规律：**工具越内建，可观测性越差**。

kernel `Python (agentic-adk)`。


In [2]:
import uuid

# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）

from common import quiet_adk_sync_warnings
quiet_adk_sync_warnings()          # 【适配】消掉 ADK 同步 API 的弃用提示
from common import adk_model, short_error, retry_call
from google.adk.agents import LlmAgent          # 【适配】书里 import Agent 却用 ADKAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

USER_ID = "user1234"

# QUESTION = "what's the latest ai news?"
QUESTION = "请介绍明日方舟塔露拉的设定和实装"

# ★ 本章新增：内建工具。不是本地函数，搜索发生在 Google 侧。
root_agent = LlmAgent(
    name="basic_search_agent",
    model=adk_model(),                          # 【适配】gemini-2.0-flash-exp 已下线
    description="Agent to answer questions using Google Search.",
    instruction="I can answer your questions by searching the internet. Just ask me anything!",
    tools=[google_search],
)

runner = InMemoryRunner(root_agent)
session_id = str(uuid.uuid4())
# 【适配】create_session 是协程 → _sync 版；书里那句 asyncio.run 在 notebook 里会冲突
runner.session_service.create_session_sync(
    app_name=runner.app_name, user_id=USER_ID, session_id=session_id
)


def text_of(content) -> str:                    # 【适配】Content 没有 .text 属性
    if getattr(content, "text", None):
        return content.text
    return "".join(p.text for p in (getattr(content, "parts", None) or [])
                   if getattr(p, "text", None))


def collect():
    return list(runner.run(
        user_id=USER_ID,
        session_id=session_id,
        new_message=types.Content(role="user", parts=[types.Part(text=QUESTION)]),
    ))


try:
    events = retry_call(collect, label="搜索 ")
    answered = 0
    for event in events:
        if event.is_final_response() and event.content:
            answered += 1
            print("Agent Response:", text_of(event.content).strip())

            # 内建工具调没调，代码里看不到，只能靠 grounding 判断
            gm = getattr(event, "grounding_metadata", None)
            print("[grounding] 搜索词:", list(getattr(gm, "web_search_queries", None) or []) if gm else "(无)")
            print("[grounding] 引用来源:", len(getattr(gm, "grounding_chunks", None) or []) if gm else 0)
    if answered == 0:
        # 中转站偶发返回空回合；不提示的话这里会一声不响 —— 别让它静默
        print(f"⚠️ 收到 {len(events)} 个事件，但没有 final response。重跑这一格即可。")
except Exception as e:
    print(f"❌ {type(e).__name__}: {short_error(e)}")


Agent Response: 在《明日方舟》中，“塔露拉”是贯穿游戏主线、最具传奇色彩和高人气的核心角色之一。从最初的宿敌、暴君，到历经洗礼后的同行者，她的设定与实装经历在玩家群体中引发了极大的关注。

以下是关于塔露拉的**剧情设定**与**干员实装**的详细介绍：

---

### 一、 角色设定（Lore & Background）

#### 1. 身世与血统
*   **本名**：塔露拉·雅特利亚斯（Talulah Artorius）。
*   **种族**：**德拉克**（Draco），拥有维多利亚王室的古老红龙血统。
*   **家庭关系**：她是维多利亚爱德华·雅特利亚斯与龙门执政官魏彦吾之妹的女儿，也是干员“陈”同母异父的姐姐。

#### 2. 命运的转折与整合运动的兴衰
*   **黑蛇的阴影**：幼年时，塔露拉被乌萨斯的阴谋家——“不死的黑蛇”科西切公爵强行拐走并收为养女。科西切在其灵魂中种下了“不死的黑蛇”继承诅咒。
*   **理想的点燃**：为了反抗科西切的残暴和对感染者的压迫，塔露拉愤而刺杀科西切，逃往乌萨斯冰原，并在此结识了阿丽娜、霜星、爱国者等人。她出于高尚的利他主义建立了**整合运动**，致力于为大地上受苦的感染者谋求出路。
*   **沦为暴君**：挚友阿丽娜被残忍杀害，以及一系列针对感染者的残酷暴行，成为了压垮塔露拉理智的最后一根稻草。在信念动摇的瞬间，科西切的诅咒生效，黑蛇成功夺取了她的身体控制权。在黑蛇的操纵下，塔露拉变成了一个冷酷的暴君，并一手策划了切尔诺伯格天灾暴动。

#### 3. 救赎与重生
*   在主线第八章《怒号光明》的控制塔决战中，陈与阿米娅合力击败了塔露拉，并利用“魔王”的力量斩断了黑蛇的意识控制，使其重新找回自我。
*   随后，她作为囚犯被罗德岛收容，后在主线第九章被九带领的新整合运动劫走。在维多利亚事件（第十至十四章）中，她以自己的方式对抗特雷西斯的军事委员会，并彻底与科西切清算，决定为自己犯下的罪孽寻找救赎。

---

### 二、 干员实装（Operator Playability）

在玩家经历了长达六年多的等待后，塔露拉终于在 **2026年11月1日** 游戏 **7.5周年“感谢庆典”** 活动（伴随SideStory「时间啊，向前！」）中作为**限定干员**正式实装上岛。

